In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import plotly.graph_objects as go

df = pd.read_csv('C:\\Users\\ARUN\\Downloads\\apple_jobs_cleaned.csv')
df = df.drop_duplicates().fillna('')

def extract_years(text):
    match = re.search(r'(\d+)\+?\s*(?:-|to)?\s*\d*\s*years?', str(text), re.IGNORECASE)
    return int(match.group(1)) if match else 0

def extract_education(text):
    text = str(text).upper()
    if 'PHD' in text: return 'PhD'
    if 'MS' in text or 'MASTER' in text: return 'Masters'
    if 'BS' in text or 'BACHELOR' in text: return 'Bachelors'
    return 'Not Specified'

def categorize_role(title):
    title = str(title).lower()
    if 'software' in title or 'developer' in title or 'ios' in title: return 'Software Engineering'
    if 'hardware' in title or 'silicon' in title or 'asic' in title: return 'Hardware Engineering'
    if 'data' in title or 'machine learning' in title or 'ai' in title: return 'Data & AI'
    if 'manager' in title or 'director' in title or 'lead' in title: return 'Management & Other'
    return 'Other'

df['years_exp'] = df['minimum_qual'].apply(extract_years)
df['edu_level'] = df['education&experience'].apply(extract_education)
df['role_category'] = df['title'].apply(categorize_role)

skills_to_track = ['Python', 'C++', 'Java', 'Swift', 'SQL', 'Machine Learning', 'Linux', 'JavaScript']
skill_counts = {}

for skill in skills_to_track:
    if skill == 'C++':
        df[skill] = df['minimum_qual'].apply(lambda x: 1 if re.search(r'c\+\+|cpp', str(x).lower()) else 0)
        skill_counts[skill] = df[skill].sum()
    else:
        df[skill] = df['minimum_qual'].apply(lambda x: 1 if re.search(r'\b' + re.escape(skill.lower()) + r'\b', str(x).lower()) else 0)
        skill_counts[skill] = df[skill].sum()

ml_df = df[df['role_category'].isin(['Software Engineering', 'Hardware Engineering', 'Data & AI', 'Management & Other'])]
X = ml_df['responsibilities'] + " " + ml_df['minimum_qual']
y = ml_df['role_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(stop_words='english', max_features=1500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

role_counts = df['role_category'].value_counts()
skill_df = pd.DataFrame(list(skill_counts.items()), columns=['Skill', 'Count']).sort_values(by='Count', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=skill_df['Count'],
    y=skill_df['Skill'],
    orientation='h',
    visible=True,
    marker=dict(color='#4C78A8')
))

fig.add_trace(go.Pie(
    labels=role_counts.index,
    values=role_counts.values,
    hole=0.4,
    visible=False,
    marker=dict(colors=['#4C78A8', '#F58518', '#E45756', '#72B7B2'])
))

fig.update_layout(
    title="Top Requested Technical Skills",
    title_x=0.5,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=0,
            x=0.5,
            y=1.15,
            xanchor="center",
            yanchor="top",
            buttons=list([
                dict(label="Top Technical Skills",
                     method="update",
                     args=[{"visible": [True, False]},
                           {"title": "Top Requested Technical Skills"}]),
                dict(label="Role Distribution",
                     method="update",
                     args=[{"visible": [False, True]},
                           {"title": "Distribution of Apple Job Categories"}])
            ])
        )
    ]
)

fig.write_html("apple_jobs_full_analysis.html")
fig.show()

Classification Report:

                      precision    recall  f1-score   support

           Data & AI       0.78      0.23      0.36        30
Hardware Engineering       1.00      0.45      0.62        11
  Management & Other       0.77      0.65      0.71        46
Software Engineering       0.79      0.97      0.87       149

            accuracy                           0.79       236
           macro avg       0.83      0.58      0.64       236
        weighted avg       0.79      0.79      0.76       236

